# How Readers Arrive at English Wikipedia: Changes in Referral Routes (2017 to 2026)

*BDCC Lab 2 · Exploratory data analysis of the English Wikipedia clickstream, November 2017 to March 2026*

## Abstract

> ✏️ **TODO after the full run (one paragraph, about 150 to 200 words).** Keep this order, because the rubric checks each part:
> **context** (Wikipedia depends on search engines and other sites for its readers);
> **problem** (the question in Section 1.2);
> **method** (77 monthly English clickstream files processed in Spark; every referrer mapped to one of five routes; shares tracked by month and by popularity cohort; the change in search share between matched months of 2018 and 2025 split into its sources; every result rerun under five handling choices);
> **results** (the numbers printed under Tables 17 to 21);
> **recommendation** (one concrete action from Section 11).

**Keywords:** Wikipedia clickstream, referral traffic, search engines, Apache Spark, shift-share decomposition, exploratory data analysis

**Setup.** The next cell loads the supporting package `clickroutes/`, which sits next to this notebook and holds every heavy query, check and figure, so this notebook keeps only the results and the text that explains them.

In [ ]:
import shutil
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from clickroutes import (checks, cohorts, config, decomposition, explore,
                         plots, preparation, routes, session, timing)
from clickroutes.report import show

warnings.filterwarnings('ignore', category=FutureWarning)
plots.apply_style()

## 1 Introduction

### 1.1 Motivation

Most readers do not start on Wikipedia. External search engines are the most common way readers reach an article, and links from other articles come second (Piccardi et al., 2023). Wikipedia's readership therefore depends on platforms it does not control: when Google began to show Wikipedia content directly on its results pages, experiments found that this reduced traffic to Wikipedia (McMahon et al., 2017). The dependence is now a live concern. Search pages increasingly answer a question with a generated summary, and in a study of 900 U.S. adults, Google users clicked a traditional result in 8% of visits when an AI summary appeared, against 15% when none appeared (Chapekis & Lieb, 2025). In October 2025 the Wikimedia Foundation reported that human pageviews from March to August 2025 were about 8% lower than in the same months of 2024 and linked the drop to generative AI and social media (Miller, 2025).

Totals alone cannot show whether readers are reaching Wikipedia differently. Total views can fall while the mix of routes holds steady, which means fewer readers arriving the same way, and the mix can shift while totals hold. The **composition** of arrival routes is therefore the quantity that measures how much Wikipedia relies on outside platforms. It matters to Wikipedia, whose readers are also the pool its editors and donors come from, and to any site that depends on referrals from search engines and other platforms.

### 1.2 Problem statement

**How has the composition of the routes by which readers arrive at English Wikipedia articles changed between November 2017 and March 2026?**

Answering it requires four things:

1. how the shares of search, external-site, other-Wikimedia, unattributed and article-link arrivals, and their per-day volumes, moved from month to month (Section 7.1);
2. whether the 100 most-read articles and the long tail moved together (Section 7.2);
3. how much of the change in search share comes from readers reaching the same articles differently versus moving to different articles (Section 7.3);
4. whether these findings hold when small pairs, the home page and unattributed traffic are handled differently (Section 8).

### 1.3 Scope

This is an exploratory study. The data are monthly counts without reader identifiers or timestamps, so the report describes **how** and **where** the mix changed; it does not attribute any change to a cause. March 2020 (COVID-19) and November 2022 (the release of ChatGPT) are marked on the time charts as reference points only.

## 2 Related work

Wulczyn and Taraborelli (2015) released the first clickstream extracts, and Wikimedia now publishes them every month (Wikimedia Research, n.d.). Piccardi et al. (2023) used one month of complete English server logs to describe how readers reach articles and move between them, and found that browsing is embedded in sessions that move to and from the external web, often through search engines. McMahon et al. (2017) showed the two-way dependence between Wikipedia and Google: Google's results are worse without Wikipedia content, and showing that content on the results page reduces visits to Wikipedia. Recent reports describe fewer clicks from search pages that carry AI summaries (Chapekis & Lieb, 2025) and fewer human pageviews to Wikipedia (Miller, 2025). None of these tracks the **mix** of arrival routes over a long window, which is the gap this study addresses with the public clickstream.

The methods are standard tools: Apache Spark and Spark SQL for distributed processing (Armbrust et al., 2015; Zaharia et al., 2016), the shift-share decomposition used in productivity research to separate change within units from reallocation between them (Griliches & Regev, 1995), and the modified z-score for flagging outliers (Iglewicz & Hoaglin, 1993).

## 3 Data source

### 3.1 What the data is

The Wikipedia clickstream contains counts of (referrer, resource) pairs extracted from Wikipedia's request logs (Wikimedia Research, n.d.; Wulczyn & Taraborelli, 2015). Each row says how many times in one month readers requested the article `curr` while their browser reported the page `prev` as the referrer. Monthly files are published on Wikimedia Downloads for several language editions; this study uses only English Wikipedia (`enwiki`).

On Jojie the files sit in `/mnt/data/public/wikipedia/clickstream/clickstream/<YYYY-MM>/clickstream-enwiki-<YYYY-MM>.tsv.gz`. The folder also holds the other language editions and two HTML pages (`index.html` and `readme.html`), which are skipped. The files are tab-separated, have no header and contain the four columns in Table 1.

**Table 1.** Columns of the raw files, as documented by Wikimedia Research (n.d.).

| Column | Type | Meaning |
|---|---|---|
| `prev` | text | The referrer, mapped to an English article title or to one of `other-search` (an external search engine), `other-external` (any other external site), `other-internal` (another Wikimedia project), `other-empty` (no referrer sent) and `other-other` (anything else) |
| `curr` | text | Title of the requested article |
| `type` | text | `link` if `prev` is an article that links to `curr`; `external` if the referrer is not English Wikipedia; `other` if both are articles but `prev` has no link to `curr` (for example internal search or a spoofed referrer) |
| `n` | integer | Number of requests for the pair in the month |

### 3.2 How Wikimedia collects it

According to the dataset's documentation (Wikimedia Research, n.d.):

- Only requests for articles in the main namespace are counted, from desktop and mobile.
- Wikimedia **attempts** to exclude automated (spider) traffic by classifying user agents and applying Wikipedia-specific filters, so some automated traffic can remain.
- Referrer URLs are mapped to the fixed values in Table 1, and requests for redirect pages are counted under the article they redirect to.
- Pairs seen too few times in a month are removed. The documentation says pairs with 10 or fewer observations are dropped; Section 6.3 checks the smallest `n` in every month, because the release threshold may have changed.
- The documentation names suspected causes of **empty referrers**: unidentified bots, browser bugs, VPNs, Wikipedia set as the browser's home page, and readers moving from mobile apps to the browser. Home-page settings are why `Main_Page` is set aside in this study.

Because rows are already aggregated by month, the data hold no reader IDs, sessions or timestamps: a row records only the last step before a page view.

## 4 Methodology

### 4.1 Overview

The pipeline has two stages (Figure 1). **Preparation** runs once in Spark: it reads the monthly files, stages each batch of months as Parquet, profiles the raw rows, maps every referrer to one of five routes and keeps one table with each article's requests by route and month. **Analysis** then runs Spark SQL on that table to answer the four parts of the problem: the route mix over time (Q1), the mix within popularity cohorts (Q2), the sources of the change in search share (Q3) and the checks on every result (Q4).

In [ ]:
fig = plots.pipeline_diagram()
plots.save(fig, 'fig01_pipeline')
plt.show()

**Figure 1.** Pipeline from the raw monthly files to the answer. Preparation runs once and keeps only a month × article × route table; every analysis (Q1 to Q4) is Spark SQL on that table, so the gzip files are decompressed exactly once.

### 4.2 Reading and storage

- **Reading.** Each file is read with an explicit four-column schema (`prev`, `curr`, `type` as text and `n` as a 64-bit integer), tab separation, no header and quoting disabled. The explicit schema avoids an inference pass over 30 GB, disabled quoting stops titles that contain quotation marks from merging rows, and a corrupt-record column keeps the raw text of any malformed line so that it is counted and inspected instead of silently dropped. The month comes from the file name.
- **Storage.** Gzip files cannot be split, so one Spark task must read each whole file. Several months are therefore read in parallel, one task per file, and written once as Parquet (the *stage*). Every check that needs raw rows runs on the stage, and each month is then aggregated into the month × article × route table. In the game plan's test, one month grew from 482 MB gzipped to 814 MB as Parquet (1.7×), so pair-level Parquet for all 77 months would need about 51 GB, above the 50 GB storage limit. The stage is therefore deleted after each batch, which is the plan's fallback of aggregating months directly into article-by-route tables. The work folder's size is checked after every month and the run stops at 40 GB. Finished months are recorded, so an interrupted run resumes where it stopped.

### 4.3 Route mapping

One `CASE` expression on `prev`, printed below, defines the routes for every analysis (Table 2). English Wikipedia titles always begin with a capital letter, so no article can start with the lowercase `other-` prefix of the mapped labels. Any `other-` label outside the five documented ones becomes *unmapped* and is counted and excluded. Article referrers of type `other` (no link, such as internal search) stay in the article route, as in the plan; their share is shown in Table 12. The mapping is checked against the labels seen in every month and against a cross-tabulation of `prev` by `type` (Section 6.5).

**Table 2.** Route assigned to each value of `prev`.

| `prev` | Route |
|---|---|
| `other-search` | Search engine |
| `other-external` | External site |
| `other-internal` | Other Wikimedia |
| `other-empty`, `other-other` | Unattributed |
| an article title | Article link |

In [ ]:
print(routes.ROUTE_CASE_SQL)

### 4.4 Filtering

Each filtering step is applied only after the counts that justify it, and Section 6.8 reports the rows and requests removed at every step.

1. **Malformed or incomplete rows** (wrong number of fields, an unparseable `n`, or a missing `prev`, `curr` or `n`) are dropped after they are counted per month and inspected (Section 6.4), because they cannot be given a route or a count.
2. **Unexpected referrer labels** are dropped, because no route is defined for them.
3. **`Main_Page` as the requested article** is set aside, because home-page settings are a documented cause of empty referrers (Wikimedia Research, n.d.). Section 6.7 measures how much of its traffic is unattributed, and Section 8 reruns every result with it included.
4. **Duplicate (`prev`, `curr`) pairs** within a month are counted (Section 6.6) but kept: their counts are added, and all their rows map to the same route because the route depends on `prev` only.

Counts are then divided by the days in the month, because months of 28 to 31 days would otherwise add a false monthly cycle to every series.

### 4.5 Popularity cohorts

Ranking every title would need a global sort of the whole table, so the 10,000 most-read titles are found with `ORDER BY ... LIMIT` and joined back to the data by broadcast, giving fixed cohorts of ranks 1–100, 101–1,000 and 1,001–10,000; all other titles form the long tail. Titles are ranked by their requests per day summed over the window, with `Main_Page` excluded. Cohorts are fixed once so that articles cannot drift between groups as their traffic changes. Because a whole-window ranking favours articles that became popular late, a second ranking on 2018 alone is used as a check.

### 4.6 Where the change in search share comes from

The early and late periods are the calendar months present in both 2018 and 2025, which removes seasonality. For each article $a$, let $w_a$ be its share of all requests in a period and $s_a$ its search share, with 0 marking 2018 and 1 marking 2025. A full outer join on title keeps every request. With $M$ the titles present in both periods, $E$ those only in 2025, $X$ those only in 2018, and $\bar S = (S_0 + S_1)/2$ the mean of the two overall search shares, the change splits exactly into four terms (Griliches & Regev, 1995):

$$
\Delta S = \underbrace{\sum_{a \in M} \bar w_a\,(s_{a1}-s_{a0})}_{\text{within}}
+ \underbrace{\sum_{a \in M} (w_{a1}-w_{a0})(\bar s_a-\bar S)}_{\text{between}}
+ \underbrace{\sum_{a \in E} w_{a1}(s_{a1}-\bar S)}_{\text{entry}}
- \underbrace{\sum_{a \in X} w_{a0}(s_{a0}-\bar S)}_{\text{exit}}
$$

where $\bar w_a$ and $\bar s_a$ are averages of the two periods. **Within** is readers reaching the same articles more or less through search, a change in *how* people find a topic. **Between** is readers shifting toward articles that depend more or less on search than average, a change in *which* topics they seek. **Entry** and **exit** are titles appearing and disappearing. Centring on $\bar S$ makes each term zero when its group looks like the average. Renamed pages break the match on title, so the share of requests covered by matched titles is reported.

### 4.7 Checks

- **Truncation.** Small pairs are removed by the release threshold, so the number of surviving small pairs rises and falls with traffic and could create a trend by itself, mostly in the long tail. Every result is recomputed with a common floor of 50 and of 100 requests per pair in every month.
- **Handling choices.** Every result is also recomputed with `Main_Page` included, without unattributed traffic (shares of attributed requests only) and with the 2018 ranking. A result is called **robust** only if its direction and rough size survive every variant, made operational as the same sign and a size between 0.5 and 2 times the baseline. If search share over all requests and over attributed requests move differently, the growth of unattributed traffic is itself a finding, because a referrer hidden by a browser or app looks like a reader lost to search.
- **Release changes.** A one-month jump in the number of pairs, in requests per day, in the smallest `n` or in the set of labels is treated as a possible change in Wikimedia's release, not in readers. Jumps are flagged when the modified z-score of the month-to-month change exceeds 3.5 (Iglewicz & Hoaglin, 1993), comparing consecutive months only so that a gap is never read as a jump.
- **Reconciliation.** Route totals must equal article totals in every month, the article table must match the usable raw rows, monthly shares must sum to 100% and the decomposition terms must sum to the observed change (Section 8.2).

### 4.8 Settings

**Table 3.** Every analysis setting and the reason for it, printed from `clickroutes/config.py` so the report and the code cannot disagree.

In [ ]:
show(config.settings_table())

## 5 Computing environment

Spark runs in local mode on Jojie. The session reads the CPU and memory limits of the Jupyter container and stays below them, enables adaptive query execution, and writes spill files outside the repository.

**Table 4.** Software versions and resources used for this run.

In [ ]:
with timing.timed('Start Spark'):
    spark, spark_settings = session.start_spark()
show(session.environment_table(spark, spark_settings))

## 6 Data preparation and exploration

### 6.1 Files and coverage

The next cell searches the data folder for English monthly files and keeps those inside the study window.

In [ ]:
with timing.timed('Find files'):
    files, found = preparation.discover_files()
coverage = preparation.coverage_table(files)
missing = coverage.loc[~coverage['present'], 'month'].tolist()
print(f'English monthly files found: {len(found)} '
      f'({len(found) - len(files)} outside {config.FIRST_MONTH} to '
      f'{config.LAST_MONTH})')
print(f'Files in the window: {len(files)} '
      f'(game plan: {config.EXPECTED_FILES})')
print(f'Compressed size: {files["gz_bytes"].sum() / 2 ** 30:.1f} GB')
print(f'Missing months ({len(missing)}): {", ".join(missing)}')

In [ ]:
fig = plots.coverage_grid(coverage)
plots.save(fig, 'fig02_coverage')
plt.show()

**Figure 2.** Months of the study window that have an English clickstream file on Jojie (blue) and months missing from the archive (grey). Missing months stay gaps in every later figure; nothing is interpolated.

### 6.2 Conversion run

The next cell is the only one that reads the raw files. On the first run it stages, profiles and aggregates every month, printing one line per batch with the work folder's size and the projected size of the kept table; expect a few hours for all months. It stops on its own before the storage limit. On later runs it finds every month done and returns at once, so *Restart and Run All* stays fast.

In [ ]:
with timing.timed('Convert: stage, profile, aggregate'):
    conversion = preparation.convert_all(
        spark, files, codec=spark_settings['parquet_codec'])

**Table 5.** Size and time of the conversion. The pair-level Parquet size is measured on the temporary stage of each month; only the month × article × route table is kept.

In [ ]:
log = preparation.conversion_log()
seconds = log[['stage_seconds', 'profile_seconds',
               'aggregate_seconds']].to_numpy().sum()
show(pd.DataFrame([
    ('Months converted', len(log)),
    ('Compressed input (GB)', log['gz_mb'].sum() / 1024),
    ('Pair-level Parquet if every month were kept (GB)',
     log['stage_mb'].sum() / 1024),
    ('Growth from gzip to Parquet (x)',
     log['stage_mb'].sum() / log['gz_mb'].sum()),
    ('Kept month x article x route table (GB)',
     log['article_table_mb'].sum() / 1024),
    ('Stage, profile and aggregate time (minutes)', seconds / 60),
], columns=['measure', 'value'], dtype=object))

### 6.3 The raw data

All statistics in this section are computed on every row of every file before any filtering (Tables 6 to 9), from the profile written during the conversion.

**Table 6.** Size of the raw data across all months.

In [ ]:
with timing.timed('Load profile tables'):
    tables = preparation.load_tables()
    monthly = explore.monthly_profile(tables)
    article_table = routes.register_article_view(spark)
    label_sets = explore.label_sets(tables['labels'])
    changes, flags = checks.release_flags(monthly, label_sets)
show(explore.raw_overview(monthly, files))

**Table 7.** Each raw column with its meaning, null count and simple statistics over all months (shares use usable rows).

In [ ]:
show(explore.column_table(monthly, tables['labels']), digits=1)

**Table 8.** Size of the data and of `n` by year. `smallest_n` is the release threshold in practice; a change in it between years is a change in Wikimedia's pipeline, not in readers.

In [ ]:
show(explore.yearly_summary(monthly), digits=2)

In [ ]:
fig = plots.volume_panels(monthly, flags)
plots.save(fig, 'fig03_volume')
plt.show()

**Figure 3.** Number of (`prev`, `curr`) pairs per month (top) and usable requests per day (bottom). Lines break at missing months. Triangles mark months whose change from the previous month is flagged as a possible release change (Section 6.9); vertical lines mark COVID-19 and ChatGPT as reference points only.

In [ ]:
cdf = explore.n_cdf(tables['n_bins'])
fig = plots.n_distribution(cdf)
plots.save(fig, 'fig04_n_distribution')
plt.show()

**Figure 4.** Cumulative share of rows and of requests in pairs up to each size, pooled over all months. Most rows are small pairs, but they carry a small part of the requests; the vertical lines mark the floors of 50 and 100 used in the truncation check.

**Table 9.** Share of usable rows and requests that a common floor of 50 or 100 requests per pair would remove, over all months and in the first and last months.

In [ ]:
show(explore.truncation_table(monthly))

> ✏️ **TODO after the run (3 to 4 sentences, Section 6.3).** Using Tables 6 to 9 and Figures 3 and 4, describe: the total rows and requests; how pairs per month and requests per day moved over the window; the smallest `n` per year and whether it matches the documented threshold of "more than 10" (Section 3.2); and the gap between Figure 4's two curves (small pairs dominate rows but not requests), which is why the truncation check matters most for the long tail.

### 6.4 Null values and malformed rows

Nulls are investigated before anything is dropped: the conversion counts, for every month, lines that did not parse into four fields or whose `n` is not a number (the corrupt-record column), null and blank values in each column, and keeps up to 50 examples of unusable rows per batch.

**Table 10.** Null, blank and malformed counts over all months, with the number of months affected.

In [ ]:
show(explore.quality_table(monthly))

**Table 11.** Examples of unusable rows, with the raw line for malformed ones (empty when there are none).

In [ ]:
show(tables['bad_rows'].head(10) if len(tables['bad_rows'])
     else pd.DataFrame({'result': ['No unusable rows in any month']}))

> ✏️ **TODO after the run (2 sentences).** If Table 10 is zero everywhere, write that no null, blank or malformed values exist in any month, so no imputation is needed. Otherwise say how many rows, in which months, what Table 11 shows about why (for example a line with a missing field), and that they are dropped because a row without a referrer, title or count cannot be routed or counted (step 1 in Table 15).

### 6.5 Referrer labels and the `type` column

**Table 12.** Requests by referrer label and `type`, as a percentage of all usable requests, with the route each label maps to.

In [ ]:
show(explore.label_type_table(tables['labels']), digits=3)
violations = explore.type_violations(tables['labels'])
expected_months = int(label_sets['as_expected'].sum())
print(f'Months with exactly the five documented labels: '
      f'{expected_months} of {len(label_sets)}')
print(f'Rows whose type contradicts their label: '
      f'{int(violations["rows"].sum()) if len(violations) else 0}')

> ✏️ **TODO after the run (2 to 3 sentences).** From Table 12 and the two printed lines: confirm that every `other-` label appears only with type `external` and article referrers only with `link` or `other`, so `type` agrees with the route mapping; give the share of article-referred requests with type `other` (no link); and state whether every month has exactly the five documented labels.

### 6.6 Duplicate pairs

**Table 13.** (`prev`, `curr`) pairs that appear on more than one row within a month.

In [ ]:
show(explore.duplicate_table(monthly), digits=5)
examples = tables['duplicate_examples']
if len(examples):
    show(examples.head(5))

Duplicate rows are kept: their counts are added, and every row of a pair maps to the same route, so keeping them changes no share. Table 13 shows how many requests this choice touches.

### 6.7 `Main_Page`

**Table 14.** Requests for `Main_Page` per day, their share of all usable requests, the share with an empty referrer, and the requests `Main_Page` sends on to articles, for the first and last months and the median month.

In [ ]:
main_page = explore.main_page_table(monthly)
median_row = main_page.median(numeric_only=True)
median_row['month'] = 'median month'
show(pd.concat([main_page.head(1), main_page.tail(1),
                median_row.to_frame().T], ignore_index=True), digits=1)

> ✏️ **TODO after the run (1 to 2 sentences).** Quote the share of `Main_Page` requests with an empty referrer and its share of all requests. A share close to 100% supports setting it aside as home-page traffic rather than reading. Links *from* `Main_Page` stay in the data as article-link arrivals.

### 6.8 Filtering

**Table 15.** Rows and requests kept at each filtering step, as counts and as percentages of the previous step and of the raw data (all months).

In [ ]:
funnel = explore.funnel(monthly)
show(funnel)

### 6.9 Possible release changes

**Table 16.** Consecutive months whose change in pairs or requests per day has a modified z-score above 3.5, or whose smallest `n` or label set differs from the month before.

In [ ]:
show(flags if len(flags) else
     pd.DataFrame({'result': ['No consecutive month was flagged']}))

> ✏️ **TODO after the run (1 to 2 sentences).** Name any flagged months and say that a one-month jump is read as a change in Wikimedia's release, not in readers, so it is not interpreted in Section 7.

## 7 Results

### 7.1 Route composition over time

One pass over the article table builds a small month × cohort table (also used in Section 7.2); the shares below are computed from it in Spark SQL, which sums `n` by month and route and turns the sums into shares with `SUM(n) OVER (PARTITION BY month)`.

In [ ]:
with timing.timed('Rank titles (ORDER BY ... LIMIT)'):
    rank_full, rank_base = cohorts.build_rankings(spark)
with timing.timed('Month x cohort table (one pass)'):
    monthly_cohort = cohorts.build_monthly_cohort(spark, article_table)
print(routes.share_sql())

In [ ]:
shares = routes.route_shares(spark)
shares_attributed = routes.route_shares(spark, attributed_only=True)
early, late = decomposition.matched_months(files['month'])
assert early, ('No calendar month is present in both comparison years; '
               'change EARLY_YEAR or LATE_YEAR in clickroutes/config.py.')
print(f'Matched months ({len(early)} per period): '
      f'{", ".join(m[5:] for m in early)} of {config.EARLY_YEAR} '
      f'and {config.LATE_YEAR}')

In [ ]:
fig = plots.share_bars(shares, 'Share of arrivals by route, Main_Page '
                       'excluded')
plots.save(fig, 'fig05_route_shares')
plt.show()

**Figure 5.** Share of arrivals at English Wikipedia articles by route, per month (`Main_Page` excluded). Each column sums to 100%; empty slots are months missing from the archive.

In [ ]:
fig = plots.volume_by_route(shares)
plots.save(fig, 'fig06_route_volume')
plt.show()

**Figure 6.** Requests per day for each route and for all routes together. Each panel has its own vertical scale, because the routes differ by a factor of about 100; read the shape of each line, not its height against other panels.

In [ ]:
fig = plots.search_share_lines(
    shares[shares['route'] == 'search'],
    shares_attributed[shares_attributed['route'] == 'search'],
    shares[shares['route'] == 'unattributed'])
plots.save(fig, 'fig07_search_share')
plt.show()

**Figure 7.** Search share over all requests (blue) and over attributed requests only (orange), with the unattributed share (green). If the two search lines diverge, the growth of unattributed traffic is the finding, because a hidden referrer looks like a reader lost to search.

**Table 17.** Share of each route (%) and requests per day in the matched months of 2018 and 2025, over all requests (top) and over attributed requests only (bottom).

In [ ]:
change_all = routes.period_change(shares, early, late)
change_attributed = routes.period_change(shares_attributed, early, late)
show(change_all)
show(change_attributed)
search = change_all.set_index('route').loc['search']
print(f'Search share: {search["share_early_%"]:.1f}% '
      f'({config.EARLY_YEAR}) to {search["share_late_%"]:.1f}% '
      f'({config.LATE_YEAR}), {search["change_pp"]:+.2f} pp; '
      f'search requests per day {search["views_change_%"]:+.1f}%')

> ✏️ **TODO after the run (4 to 6 sentences, the core answer to part 1).** Using Figures 5 to 7 and Table 17: which routes gained and lost share between 2018 and 2025 (in pp), and whether per-day volumes moved with the shares (falling volume with a stable mix means fewer readers arriving the same way; a shifting mix means readers arriving differently). Say whether the search lines over all and over attributed requests tell the same story. Describe the reference months only as markers ("after November 2022 the series ..."), never as causes, and do not interpret months flagged in Table 16.

### 7.2 Popularity cohorts

Shares are recomputed inside each fixed cohort to see whether the most-read articles and the long tail moved together.

In [ ]:
cohort_shares = routes.route_shares(spark, group='cohort')
concentration = cohorts.concentration(monthly_cohort)
fig = plots.cohort_panels(cohort_shares, concentration)
plots.save(fig, 'fig08_cohorts')
plt.show()

**Figure 8.** Route shares within each popularity cohort (ranks fixed on whole-window requests per day, `Main_Page` excluded), and each cohort's share of all arrivals (bottom right). Darker lines are more popular cohorts.

**Table 18.** Change in each route's share between the matched months of 2018 and 2025, by cohort (percentage points), and each cohort's share of all arrivals.

In [ ]:
change_cohort = routes.period_change(cohort_shares, early, late,
                                     by=('cohort', 'route'))
pivot = change_cohort.pivot(index='cohort', columns='route',
                            values='change_pp')
pivot = pivot[list(config.ROUTES)].rename_axis(columns=None).reset_index()
pivot = cohorts.cohort_order(pivot)
held = concentration[concentration['month'].isin(early + late)].copy()
held['period'] = held['month'].isin(late).map(
    {False: f'share_of_all_{config.EARLY_YEAR}_%',
     True: f'share_of_all_{config.LATE_YEAR}_%'})
held = (held.groupby(['cohort', 'period'])['views_per_day'].sum()
        .unstack('period'))
held = 100 * held / held.sum()
show(pivot.merge(held.reset_index(), on='cohort'))

> ✏️ **TODO after the run (3 to 4 sentences, part 2).** Did the top 100 and the long tail move in the same direction on search, and by similar amounts? Which cohort carries the largest change? Did the most-read articles take a larger or smaller share of all arrivals? Remind the reader that the long tail is most affected by truncation, so its result counts only if it survives the floors of 50 and 100 in Section 8.

### 7.3 Where the change in search share comes from

The query below joins the 2018 and 2025 periods on title with a full outer join and sums the four terms of Section 4.6.

In [ ]:
with timing.timed('Decomposition (all variants)'):
    terms = {name: decomposition.decompose(spark, early, late, name)
             for name in decomposition.VARIANTS}
summaries = pd.concat([decomposition.summarize(t, name)
                       for name, t in terms.items()], ignore_index=True)
baseline = summaries.iloc[[0]]

**Table 19.** Decomposition of the change in search share between the matched months of 2018 and 2025 (percentage points). The residual is the observed change minus the four terms and should be zero; the last columns give the share of requests covered by titles present in both periods.

In [ ]:
show(baseline.T.reset_index().set_axis(['measure', 'value'], axis=1),
     digits=3)

In [ ]:
fig = plots.decomposition_bars(baseline)
plots.save(fig, 'fig09_decomposition')
plt.show()

**Figure 9.** The change in search share (grey) and its four sources: the same articles reached differently (within), readers shifting toward articles that rely more or less on search (between), and titles appearing or disappearing. Red bars lower the search share and blue bars raise it.

**Table 20.** Within and between terms by popularity cohort (percentage points); rows sum to the terms in Table 19.

In [ ]:
show(decomposition.by_cohort(terms['baseline']), digits=3)

> ✏️ **TODO after the run (3 to 5 sentences, part 3).** From Table 19 and Figure 9: how much of the change is *within* (the same articles reached less or more through search, a change in how people find topics) versus *between* (readers moving to articles that rely more or less on search, a change in which topics they seek), and how large entry and exit are. Quote the matched share of requests. From Table 20, say which cohort contributes most of the within term.

## 8 Reliability checks

### 8.1 Do the findings survive other handling choices?

Each headline finding is recomputed with `Main_Page` included, with attributed requests only, with the 2018 ranking, and with floors of 50 and 100 requests per pair.

In [ ]:
with timing.timed('Robustness reruns'):
    robust = checks.robustness(spark, early, late)
fig = plots.robustness_dots(robust)
plots.save(fig, 'fig10_robustness')
plt.show()

**Figure 10.** The change between 2018 and 2025 in four headline findings under each handling choice (black: baseline). A finding is robust when every dot stays on the baseline's side of zero at a similar distance.

**Table 21.** Verdict for each finding across all variants (robust = same sign and 0.5 to 2 times the baseline in every variant).

In [ ]:
show(checks.robustness_summary(robust))

**Table 22.** Every finding under every variant, with its verdict.

In [ ]:
show(robust)

> ✏️ **TODO after the run (3 to 4 sentences, part 4).** Which findings are robust, which keep their direction but change size, and which reverse? Give special attention to the long tail under the floors of 50 and 100, and to search share with attributed requests only: if that variant shrinks the decline a lot, say that part of the apparent loss of search traffic is a growth in unattributed traffic.

### 8.2 Reconciliation

**Table 23.** Reconciliation checks. The first two rows compare with the game plan's counts and are informational; every other row must pass.

In [ ]:
with timing.timed('Reconciliation checks'):
    integrity = checks.integrity_checks(
        spark, monthly, monthly_cohort, funnel, files, shares, summaries,
        violations)
show(integrity, digits=6)

### 8.3 Articles that receive almost only unattributed traffic

The documentation lists unidentified bots among the suspected causes of empty referrers (Wikimedia Research, n.d.). Articles with at least 1,000 requests per day of which at least 90% are unattributed are a sign of such traffic.

**Table 24.** Number of such articles per month and their share of all unattributed requests (first and last months), and the largest of them in the last month.

In [ ]:
with timing.timed('Unattributed-heavy articles'):
    heavy_by_month, heavy_latest = explore.unattributed_heavy(spark)
show(pd.concat([heavy_by_month.head(3), heavy_by_month.tail(3)]), digits=3)
if len(heavy_latest):
    show(heavy_latest, digits=3)
else:
    print('No article meets both thresholds in the last month.')

> ✏️ **TODO after the run (1 to 2 sentences).** If the count or share in Table 24 grows over time, part of the unattributed growth sits in a few pages that readers rarely visit on purpose, which supports reporting search share over attributed requests as well.

## 9 Discussion

### 9.1 Answer to the problem

> ✏️ **TODO after the run (one paragraph).** Answer the question of Section 1.2 directly in the first sentence, for example: "Between 2018 and 2025 the share of arrivals from search engines fell (or rose) by X pp while ... ". Then summarise parts 1 to 4 in one sentence each, quoting Tables 17 to 21, and state which parts are robust.

### 9.2 What it means for Wikipedia

> ✏️ **TODO after the run (one paragraph).** Connect the robust findings to the motivation: is Wikipedia relying more or less on search engines to bring readers, and is the change concentrated in a few popular articles or spread across the long tail? Keep the language descriptive (the data show *how* the mix changed, not *why*).

### 9.3 Beyond this study

The method applies to any platform that depends on referral traffic. Tracking the mix of arrival routes, not only total visits, separates fewer readers arriving the same way from readers arriving differently. Reporting shares over all and over attributed traffic shows how much of an apparent loss is only a loss of attribution, and the within/between split tells a publisher whether readers are finding the same pages differently or moving to different pages, which call for different responses (for example, changing how pages appear in search versus changing which pages are written). The same checks (common floors on small counts, fixed cohorts, reconciliation of totals) apply to any truncated, aggregated log.

### 9.4 Threats to validity

- **Missing months.** Gaps are never interpolated, so the comparison rests on matched months that exist in both years.
- **Truncation.** Small pairs are removed by the release threshold, which affects the long tail most; the floors of 50 and 100 test this.
- **Unattributed traffic.** Empty referrers mix home-page use, apps, privacy settings and bots, so they are never reassigned to a route.
- **Release changes.** Wikimedia's bot detection and referrer mapping can change between releases; flagged months are not interpreted.
- **Titles.** Renamed pages count as one exit and one entry, which inflates those terms; the matched share in Table 19 bounds the effect.

## 10 Conclusion

> ✏️ **TODO after the run (one paragraph, 4 to 6 sentences).** Restate the question, give the answer with the two or three most important numbers (the change in search share in pp; the within versus between split; whether the top 100 and the tail agree), say which findings are robust, and end with the single most useful implication.

## 11 Recommendations and future work

**Limitations.** The clickstream is monthly and aggregated, so it cannot show sessions, individual readers or the order of steps, and every finding describes change without explaining it. Twenty-four months are missing from the archive. Pairs below the release threshold are removed, so the long tail is measured least precisely. The origin of unattributed requests is unknown, and some automated traffic may remain despite Wikimedia's filters. Only English Wikipedia is studied.

**Future work.**

1. Compare the monthly route totals with Wikimedia's published pageview counts by access method (desktop, mobile web, apps) to see whether the growth of unattributed traffic follows a shift toward apps.
2. Repeat the analysis for the other language editions in the same folder to see whether the change is global or specific to English.
3. Mark known changes in browser referrer policies on the time series to test whether jumps in unattributed traffic line up with them.
4. Extend the decomposition to the article-link route to see whether readers who arrive from search now move on to fewer articles.
5. Group articles by topic (for example with article categories) to see whether the change in search share is concentrated in topics that search pages now answer directly.

> ✏️ **TODO after the run (one sentence).** Add the one recommendation the abstract ends with, chosen from the robust findings.

## 12 Reproducibility

The notebook runs from top to bottom with *Kernel → Restart Kernel and Run All Cells*. The data folder is read-only and every intermediate table is written to a private work folder outside the repository, so no data file is committed. The conversion records finished months and resumes after an interruption; small result tables are cached and rebuilt automatically when the converted data or the settings change. No step uses random numbers, and every ranking breaks ties by title, so reruns give identical results. No username, home path or token appears in any output.

**Table 25.** The slowest steps of this run.

In [ ]:
show(timing.timing_table())

The last cell stops Spark and removes temporary files. The work folder (the kept month × article × route table and the profile tables) is kept by default so the next run skips the conversion; set `DELETE_WORK_DIR_AT_END = True` in `clickroutes/config.py` for the final run if the space is needed.

In [ ]:
spark.stop()
shutil.rmtree(config.STAGE_DIR, ignore_errors=True)
shutil.rmtree(spark_settings['spill_path'], ignore_errors=True)
if config.DELETE_WORK_DIR_AT_END:
    shutil.rmtree(config.WORK_DIR, ignore_errors=True)
    print('Deleted the private work folder.')
else:
    size = preparation.dir_size(config.WORK_DIR) / 2 ** 30
    print(f'Kept the work folder {config.display_path(config.WORK_DIR)} '
          f'({size:.1f} GB) for the next run.')

## References

Armbrust, M., Xin, R. S., Lian, C., Huai, Y., Liu, D., Bradley, J. K., Meng, X., Kaftan, T., Franklin, M. J., Ghodsi, A., & Zaharia, M. (2015). Spark SQL: Relational data processing in Spark. In *Proceedings of the 2015 ACM SIGMOD International Conference on Management of Data* (pp. 1383–1394). Association for Computing Machinery. https://doi.org/10.1145/2723372.2742797

Chapekis, A., & Lieb, A. (2025, July 22). *Google users are less likely to click on links when an AI summary appears in the results*. Pew Research Center. https://www.pewresearch.org/short-reads/2025/07/22/google-users-are-less-likely-to-click-on-links-when-an-ai-summary-appears-in-the-results/

Griliches, Z., & Regev, H. (1995). Firm productivity in Israeli industry 1979–1988. *Journal of Econometrics, 65*(1), 175–203.

Iglewicz, B., & Hoaglin, D. C. (1993). *How to detect and handle outliers*. ASQC Quality Press.

McMahon, C., Johnson, I., & Hecht, B. (2017). The substantial interdependence of Wikipedia and Google: A case study on the relationship between peer production communities and information technologies. *Proceedings of the International AAAI Conference on Web and Social Media, 11*(1), 142–151. https://doi.org/10.1609/icwsm.v11i1.14883

Miller, M. (2025, October 17). *New user trends on Wikipedia*. Diff. https://diff.wikimedia.org/2025/10/17/new-user-trends-on-wikipedia/

Piccardi, T., Gerlach, M., Arora, A., & West, R. (2023). A large-scale characterization of how readers browse Wikipedia. *ACM Transactions on the Web, 17*(2), Article 11. https://doi.org/10.1145/3580318

Wikimedia Research. (n.d.). *Research:Wikipedia clickstream*. Meta-Wiki. Retrieved November 4, 2026, from https://meta.wikimedia.org/wiki/Research:Wikipedia_clickstream

Wulczyn, E., & Taraborelli, D. (2015). *Wikipedia clickstream* [Data set]. figshare. https://doi.org/10.6084/m9.figshare.1305770

Zaharia, M., Xin, R. S., Wendell, P., Das, T., Armbrust, M., Dave, A., Meng, X., Rosen, J., Venkataraman, S., Franklin, M. J., Ghodsi, A., Gonzalez, J., Shenker, S., & Stoica, I. (2016). Apache Spark: A unified engine for big data processing. *Communications of the ACM, 59*(11), 56–65. https://doi.org/10.1145/2934664